In [2]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

plt.rcParams['figure.figsize'] = (12,8)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 12

In [3]:
G = nx.read_edgelist('graph.edgelist')

largest_component_nodes = max(nx.connected_components(G), key=len)
G = G.subgraph(largest_component_nodes).copy()

In [6]:
ecc = nx.eccentricity(G)
ecc_centrality = {n: 1.0 / e for n, e in ecc.items() if e > 0}
print("\n--- Excentricidade ---")
for n, v in sorted(ecc_centrality.items(), key=lambda x: -x[1])[:5]:
    print(f"  Nó {n}: {v:.4f}")



--- Excentricidade ---
  Nó TERT: 0.1111
  Nó RUVBL2: 0.1111
  Nó HSP90AA1: 0.1111
  Nó DKC1: 0.1111
  Nó PIH1D1: 0.1111


In [7]:
closeness = nx.closeness_centrality(G)
print("\n--- Closeness centrality ---")
for n, v in sorted(closeness.items(), key=lambda x: -x[1])[:5]:
    print(f"  Nó {n}: {v:.4f}")



--- Closeness centrality ---
  Nó TP53: 0.2963
  Nó SRC: 0.2947
  Nó CTNNB1: 0.2910
  Nó EGFR: 0.2908
  Nó HSP90AA1: 0.2847


In [8]:
harmonic = nx.harmonic_centrality(G)
# normalizar pelo número de nós - 1
n_nodes = G.number_of_nodes()
harmonic_norm = {n: v / (n_nodes - 1) for n, v in harmonic.items()}
print("\n--- Harmonic centrality (normalizada) ---")
for n, v in sorted(harmonic_norm.items(), key=lambda x: -x[1])[:5]:
    print(f"  Nó {n}: {v:.4f}")


--- Harmonic centrality (normalizada) ---
  Nó TP53: 0.3414
  Nó SRC: 0.3343
  Nó CTNNB1: 0.3325
  Nó EGFR: 0.3277
  Nó HSP90AA1: 0.3195


In [9]:
betweenness = nx.betweenness_centrality(G, normalized=True)
print("\n--- Betweenness centrality ---")
for n, v in sorted(betweenness.items(), key=lambda x: -x[1])[:5]:
    print(f"  Nó {n}: {v:.4f}")


--- Betweenness centrality ---
  Nó TP53: 0.1612
  Nó CTNNB1: 0.1213
  Nó SRC: 0.0900
  Nó HSP90AA1: 0.0762
  Nó EGFR: 0.0706


In [10]:
pagerank = nx.pagerank(G, alpha=0.85)
print("\n--- PageRank ---")
for n, v in sorted(pagerank.items(), key=lambda x: -x[1])[:5]:
    print(f"  Nó {n}: {v:.4f}")


--- PageRank ---
  Nó TP53: 0.0028
  Nó CTNNB1: 0.0025
  Nó FN1: 0.0022
  Nó HDAC1: 0.0018
  Nó CD44: 0.0018


In [5]:
core_number = nx.core_number(G)
print("\n--- K-cores (core number por nó) ---")
for n, v in sorted(core_number.items(), key=lambda x: -x[1])[:20]:
    print(f"  Nó {n}: core={v}")


--- K-cores (core number por nó) ---
  Nó SMARCA4: core=12
  Nó ACTL6A: core=12
  Nó DPF2: core=12
  Nó SMARCC1: core=12
  Nó ARID1A: core=12
  Nó SMARCE1: core=12
  Nó ARID2: core=12
  Nó SMARCB1: core=12
  Nó PHF10: core=12
  Nó SMARCA2: core=12
  Nó SMARCD1: core=12
  Nó SMARCC2: core=12
  Nó PBRM1: core=12
  Nó BRD7: core=11
  Nó ARID1B: core=11
  Nó TP53: core=10
  Nó ACTL6B: core=10
  Nó EP300: core=10
  Nó HDAC2: core=10
  Nó H3-3B: core=10


In [12]:
clustering = nx.clustering(G)
print("\n--- Local clustering coefficient ---")
for n, v in sorted(clustering.items(), key=lambda x: -x[1])[:5]:
    print(f"  Nó {n}: {v:.4f}")


--- Local clustering coefficient ---
  Nó MVP: 1.0000
  Nó PARP4: 1.0000
  Nó INO80B: 1.0000
  Nó BRD7: 1.0000
  Nó SMARCD2: 1.0000


In [14]:
assort_degree = nx.degree_assortativity_coefficient(G)
print(f"\n--- Assortatividade de grau: {assort_degree:.4f} ---")


--- Assortatividade de grau: 0.1610 ---


In [15]:
edge_betweenness = nx.edge_betweenness_centrality(G, normalized=True)
print("\n--- Edge Betweenness (top 5 arestas) ---")
for (u, v), w in sorted(edge_betweenness.items(), key=lambda x: -x[1])[:5]:
    print(f"  ({u}, {v}): {w:.4f}")


--- Edge Betweenness (top 5 arestas) ---
  (HSP90AA1, DKC1): 0.0190
  (TP53, SRC): 0.0173
  (SRC, DLG4): 0.0156
  (EGFR, YBX1): 0.0150
  (TP53, EGFR): 0.0144


In [19]:
def mnc(G):
    scores = {}

    for v in G.nodes():
        neighborhood = list(G.neighbors(v))
        H = G.subgraph(neighborhood)

        largest_component = max(
            nx.connected_components(H),
            key=len,
            default=set()
        )

        scores[v] = len(largest_component)

    return scores


def dmnc(G):
    scores = {}

    for v in G.nodes():
        neighborhood = list(G.neighbors(v))
        H = G.subgraph(neighborhood)

        largest_component = max(
            nx.connected_components(H),
            key=len,
            default=set()
        )

        n = len(largest_component)

        if n <= 1:
            scores[v] = 0.0
        else:
            C = H.subgraph(largest_component)
            edges = C.number_of_edges()

            scores[v] = edges / (n ** 0.7)

    return scores

mnc_scores = mnc(G)
dmnc_scores = dmnc(G)

top_dmnc = sorted(
    dmnc_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]
top_mnc = sorted(
    mnc_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print(top_dmnc)
print(top_mnc)

[('TRAF6', 17.114592519067546), ('TP53', 16.642265153508838), ('MYD88', 15.48913772118118), ('HDAC1', 15.389381591955402), ('SMARCC1', 15.337947102954843), ('SMARCE1', 15.337947102954843), ('PHF10', 15.337947102954843), ('SMARCB1', 15.000725121839896), ('SMARCC2', 15.000725121839896), ('ARID1A', 14.809052375266743)]
[('TP53', 115), ('CTNNB1', 86), ('FN1', 85), ('CD4', 79), ('HDAC1', 76), ('EGFR', 70), ('SRC', 70), ('CD44', 66), ('EP300', 64), ('AKT1', 63)]
